In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!git clone https://github.com/arminestian/JamMa.git

In [ ]:
import os
os.environ["PYTHONPATH"] = "/kaggle/working/JamMa"

In [ ]:
!pip install \
    albucore \
    albumentations \
    contourpy \
    einops \
    imageio \
    imgaug \
    kornia \
    loguru \
    poselib \
    PyWavelets \
    scikit-image \
    shapely \
    tensorboard \
    thop \
    tifffile \
    timm \
    torchmetrics \
    yacs \
    --upgrade --quiet

In [ ]:
!pip install mamba_ssm --no-build-isolation --quiet
!pip install triton --quiet
!pip install causal-conv1d --quiet

In [ ]:
!pip install causal-conv1d>=1.4.0 mamba-ssm --no-build-isolation --quiet

In [ ]:
!TORCH_CUDA_ARCH_LIST="7.5" MAX_JOBS=1 pip install -v causal-conv1d>=1.4.0 --no-build-isolation

In [ ]:
updated_profiler_code = """import torch
from contextlib import contextmanager

# Robust compatibility for PyTorch Lightning Profilers
try:
    from pytorch_lightning.profilers import SimpleProfiler, PassThroughProfiler
except ImportError:
    try:
        from pytorch_lightning.profiler import SimpleProfiler, PassThroughProfiler
    except ImportError:
        from lightning.pytorch.profilers import SimpleProfiler, PassThroughProfiler

# Robust compatibility for rank_zero_only decorator
try:
    from pytorch_lightning.utilities.rank_zero import rank_zero_only
except ImportError:
    try:
        from pytorch_lightning.utilities import rank_zero_only
    except ImportError:
        from lightning.fabric.utilities.rank_zero import rank_zero_only


class InferenceProfiler(SimpleProfiler):
    \"\"\"
    This profiler records duration of actions with cuda.synchronize()
    Use this in test time. 
    \"\"\"

    def __init__(self):
        super().__init__()
        self.start = rank_zero_only(self.start)
        self.stop = rank_zero_only(self.stop)
        self.summary = rank_zero_only(self.summary)

    @contextmanager
    def profile(self, action_name: str) -> None:
        try:
            torch.cuda.synchronize()
            self.start(action_name)
            yield action_name
        finally:
            torch.cuda.synchronize()
            self.stop(action_name)


def build_profiler(name):
    if name == 'inference':
        return InferenceProfiler()
    elif name == 'pytorch':
        try:
            from pytorch_lightning.profilers import PyTorchProfiler
        except ImportError:
            try:
                from pytorch_lightning.profiler import PyTorchProfiler
            except ImportError:
                from lightning.pytorch.profilers import PyTorchProfiler
        return PyTorchProfiler(use_cuda=True, profile_memory=True, row_limit=100)
    elif name is None:
        return PassThroughProfiler()
    else:
        raise ValueError(f'Invalid profiler: {name}')
"""

file_path = "/kaggle/working/JamMa/src/utils/profiler.py"

with open(file_path, "w") as f:
    f.write(updated_profiler_code)

print("Successfully patched profiler.py with adaptive imports!")

In [ ]:
!pip install pytorch-lightning==1.8.6 --quiet

In [ ]:
%%writefile /kaggle/working/roadscene_to_megadepth.py

"""
Convert RoadScene dataset → MegaDepth-style .npz index files for JamMa training.

RoadScene structure (input):
  RoadScene/
    crop_HR_visible/   FLIR_XXXXX.jpg   (visible images, high-res)
    cropinfrared/      FLIR_XXXXX.jpg   (infrared images, same names)

Output structure (MegaDepth-style):
  data/roadscene/
    train/
      visible/         FLIR_XXXXX.jpg
      infrared/        FLIR_XXXXX.jpg
    test/
      visible/         FLIR_XXXXX.jpg
      infrared/        FLIR_XXXXX.jpg
    index/
      scene_info/      FLIR_XXXXX.npz   (one per image pair / "scene")
      trainvaltest_list/
        train_list.txt
        test_list.txt

Key design decisions
--------------------
* Each IR+visible pair is treated as ONE scene with exactly 2 images.
* The images are already aligned (identity relative pose).
* Since no depth is provided, we use a large constant depth (100 m) so
  that JamMa's warp function maps pixels nearly 1-to-1 (identity homography).
* Intrinsics are estimated from image size using a standard 60° FoV assumption.
* pair_infos overlap_score = 1.0 (fully overlapping by construction).

Usage
-----
python roadscene_to_megadepth.py \
    --roadscene_dir /path/to/RoadScene \
    --output_dir    /path/to/data/roadscene \
    --train_ratio   0.85 \
    --seed          42
"""

import os
import sys
import glob
import shutil
import argparse
import random
import numpy as np
from pathlib import Path
from PIL import Image

def estimate_intrinsics(w, h, fov_deg=60.0):
    """
    Build a 3x3 camera intrinsic matrix from image size.
    Assumes a horizontal FoV of `fov_deg` degrees (60° is a reasonable default
    for consumer cameras / dashcams).

    K = [[f,  0, cx],
         [0,  f, cy],
         [0,  0,  1]]
    """
    fov_rad = np.deg2rad(fov_deg)
    f = (w / 2.0) / np.tan(fov_rad / 2.0)
    cx, cy = w / 2.0, h / 2.0
    K = np.array([[f, 0, cx],
                  [0, f, cy],
                  [0, 0,  1]], dtype=np.float32)
    return K


def identity_pose():
    """4x4 camera-to-world pose matrix (identity = camera at world origin)."""
    return np.eye(4, dtype=np.float32)


def constant_depth_map(h, w, depth=100.0):
    """
    Return a constant depth map of shape (H, W).
    100 m makes the warp between the two aligned views essentially identity.
    """
    return np.full((h, w), depth, dtype=np.float32)


def build_scene_npz(vis_rel, ir_rel, img_w, img_h):
    """
    Build the dict that will be saved as a .npz scene file.

    MegaDepth loader expects:
      image_paths  : (N,)  relative paths from data root
      depth_paths  : (N,)  relative paths to depth maps  (None → dummy)
      intrinsics   : (N, 3, 3)
      poses        : (N, 4, 4)  camera-to-world
      pair_infos   : list of (idx0, idx1, overlap_score)
                     overlap_score ∈ [0, 1]
    """
    K = estimate_intrinsics(img_w, img_h)

    scene = {
        # Two images per scene: [visible, infrared]
        "image_paths": np.array([vis_rel, ir_rel]),
        "depth_paths": np.array([None, None]),       # no real depth
        "intrinsics":  np.stack([K, K], axis=0),     # (2, 3, 3)
        "poses":       np.stack([identity_pose(),
                                 identity_pose()], axis=0),  # (2, 4, 4)
        # One pair: (img0_idx=0, img1_idx=1, overlap=1.0)
        "pair_infos":  [(0, 1, 1.0)],
    }
    return scene


def convert(roadscene_dir: str, output_dir: str,
            train_ratio: float, seed: int):

    roadscene_dir = Path(roadscene_dir)
    output_dir    = Path(output_dir)

    vis_dir = roadscene_dir / "crop_HR_visible"
    ir_dir  = roadscene_dir / "cropinfrared"

    if not vis_dir.exists():
        sys.exit(f"[ERROR] Cannot find visible dir: {vis_dir}\n"
                 "        Make sure --roadscene_dir points at the repo root.")
    if not ir_dir.exists():
        sys.exit(f"[ERROR] Cannot find infrared dir: {ir_dir}\n"
                 "        Make sure --roadscene_dir points at the repo root.")

    # Collect matched pairs (same stem name in both dirs)
    vis_files = sorted(vis_dir.glob("*.jpg")) + sorted(vis_dir.glob("*.png"))
    all_stems = [f.stem for f in vis_files
                 if (ir_dir / f.name).exists() or
                    (ir_dir / (f.stem + ".png")).exists()]

    if len(all_stems) == 0:
        sys.exit("[ERROR] No matching IR/visible pairs found. "
                 "Check directory names.")

    print(f"[INFO] Found {len(all_stems)} matched pairs.")

    # Reproducible train/test split
    random.seed(seed)
    random.shuffle(all_stems)
    n_train = max(1, int(len(all_stems) * train_ratio))
    train_stems = all_stems[:n_train]
    test_stems  = all_stems[n_train:]
    print(f"[INFO] Split → train: {len(train_stems)}, test: {len(test_stems)}")


    for split in ("train", "test"):
        (output_dir / split / "visible").mkdir(parents=True, exist_ok=True)
        (output_dir / split / "infrared").mkdir(parents=True, exist_ok=True)
    scene_info_dir = output_dir / "index" / "scene_info"
    list_dir       = output_dir / "index" / "trainvaltest_list"
    scene_info_dir.mkdir(parents=True, exist_ok=True)
    list_dir.mkdir(parents=True, exist_ok=True)

    split_map = {"train": train_stems, "test": test_stems}

    for split, stems in split_map.items():
        scene_names = []

        for stem in stems:
            # Locate source files
            vis_src = vis_dir / f"{stem}.jpg"
            if not vis_src.exists():
                vis_src = vis_dir / f"{stem}.png"

            ir_src = ir_dir / f"{stem}.jpg"
            if not ir_src.exists():
                ir_src = ir_dir / f"{stem}.png"

            ext = vis_src.suffix  # use visible's extension for both

            # Copy images to output/<split>/
            vis_dst = output_dir / split / "visible" / f"{stem}{ext}"
            ir_dst  = output_dir / split / "infrared" / f"{stem}{ext}"
            shutil.copy2(vis_src, vis_dst)
            shutil.copy2(ir_src,  ir_dst)

            # Get image size (all pairs assumed same size; read once)
            with Image.open(vis_src) as img:
                img_w, img_h = img.size   # PIL: (width, height)

            # Relative paths from the data root (output_dir/<split>/)
            vis_rel = f"visible/{stem}{ext}"
            ir_rel  = f"infrared/{stem}{ext}"

            # Build and save scene .npz
            scene_data = build_scene_npz(vis_rel, ir_rel, img_w, img_h)
            npz_path = scene_info_dir / f"{stem}.npz"
            np.savez(npz_path, **scene_data)

            scene_names.append(stem)

        # Write list file:  one scene npz name per line (without .npz suffix)
        list_path = list_dir / f"{split}_list.txt"
        with open(list_path, "w") as f:
            for name in scene_names:
                f.write(name + "\n")

        print(f"[INFO] {split}: wrote {len(scene_names)} scenes → {list_path}")


    abs_out = output_dir.resolve()
    print("\n" + "=" * 65)
    print("SUCCESS! Now create configs/data/roadscene_trainval.py:\n")
    print(f"""\
from configs.data.base import cfg

TRAIN_BASE_PATH = "{abs_out}/index"

cfg.DATASET.TRAINVAL_DATA_SOURCE = "MegaDepth"
cfg.DATASET.TRAIN_DATA_ROOT      = "{abs_out}/train"
cfg.DATASET.TRAIN_NPZ_ROOT       = f"{{TRAIN_BASE_PATH}}/scene_info"
cfg.DATASET.TRAIN_LIST_PATH      = f"{{TRAIN_BASE_PATH}}/trainvaltest_list/train_list.txt"
cfg.DATASET.MIN_OVERLAP_SCORE_TRAIN = 0.0

TEST_BASE_PATH = "{abs_out}/index"

cfg.DATASET.TEST_DATA_SOURCE = "MegaDepth"
cfg.DATASET.VAL_DATA_ROOT  = cfg.DATASET.TEST_DATA_ROOT  = "{abs_out}/test"
cfg.DATASET.VAL_NPZ_ROOT   = cfg.DATASET.TEST_NPZ_ROOT   = f"{{TEST_BASE_PATH}}/scene_info"
cfg.DATASET.VAL_LIST_PATH  = cfg.DATASET.TEST_LIST_PATH  = f"{{TEST_BASE_PATH}}/trainvaltest_list/test_list.txt"
cfg.DATASET.MIN_OVERLAP_SCORE_TEST = 0.0

cfg.TRAINER.N_SAMPLES_PER_SUBSET = 1    # 1 pair per scene (IR+vis)
cfg.DATASET.MGDPT_IMG_RESIZE = 640
""")
    print("Then run training:")
    print("  python train.py \\")
    print("      configs/data/roadscene_trainval.py \\")
    print("      configs/jamma/outdoor.py \\")
    print("      --exp_name roadscene_finetune \\")
    print("      --ckpt_path /path/to/jamma_pretrained.ckpt \\")
    print("      --gpus 1 --batch_size 2 --max_epochs 30")
    print("=" * 65)


def parse_args():
    p = argparse.ArgumentParser(
        description="Convert RoadScene dataset to MegaDepth-style format "
                    "for JamMa fine-tuning.")
    p.add_argument("--roadscene_dir", required=True,
                   help="Root of the cloned RoadScene repo "
                        "(contains crop_HR_visible/ and cropinfrared/)")
    p.add_argument("--output_dir", required=True,
                   help="Where to write the converted dataset, "
                        "e.g. /path/to/data/roadscene")
    p.add_argument("--train_ratio", type=float, default=0.85,
                   help="Fraction of pairs used for training (default 0.85)")
    p.add_argument("--seed", type=int, default=42,
                   help="Random seed for reproducible split (default 42)")
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    convert(
        roadscene_dir=args.roadscene_dir,
        output_dir=args.output_dir,
        train_ratio=args.train_ratio,
        seed=args.seed,
    )


In [ ]:
!sed -i 's|/path/to/data/roadscene|/kaggle/working/JamMa/src/datasets/RoadScene dataset|g' \
    /kaggle/working/JamMa/configs/data/roadscene_trainval.py

In [ ]:
!sed -i \
  "s/self\.scene_info = np\.load(\(.*\))/self.scene_info = dict(np.load(\1))/" \
  /kaggle/working/JamMa/src/datasets/megadepth.py

In [ ]:
%%writefile /kaggle/working/JamMa/train.py
import math
import argparse
import pprint
from distutils.util import strtobool
from pathlib import Path
from loguru import logger as loguru_logger

import pytorch_lightning as pl
from pytorch_lightning.utilities import rank_zero_only
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.strategies import DDPStrategy

from src.config.default import get_cfg_defaults
from src.utils.misc import get_rank_zero_only_logger, setup_gpus
from src.utils.profiler import build_profiler
from src.lightning.data import MultiSceneDataModule
from src.lightning.lightning_jamma import PL_JamMa
loguru_logger = get_rank_zero_only_logger(loguru_logger)


def parse_args():
    parser = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument('data_cfg_path', type=str)
    parser.add_argument('main_cfg_path', type=str)
    parser.add_argument('--exp_name', type=str, default='default_exp_name')
    parser.add_argument('--dump_dir', type=str, default=None)
    parser.add_argument('--batch_size', type=int, default=4)
    parser.add_argument('--num_workers', type=int, default=4)
    parser.add_argument('--pin_memory', type=lambda x: bool(strtobool(x)), nargs='?', default=True)
    parser.add_argument('--ckpt_path', type=str, default=None)
    parser.add_argument('--disable_ckpt', action='store_true')
    parser.add_argument('--profiler_name', type=str, default=None)
    parser.add_argument('--parallel_load_data', action='store_true')
    parser = pl.Trainer.add_argparse_args(parser)
    return parser.parse_args()


def main():
    args = parse_args()
    rank_zero_only(pprint.pprint)(vars(args))

    config = get_cfg_defaults()
    config.merge_from_file(args.main_cfg_path)
    config.merge_from_file(args.data_cfg_path)
    pl.seed_everything(config.TRAINER.SEED)
    args.gpus = _n_gpus = setup_gpus(args.gpus)
    config.TRAINER.WORLD_SIZE = _n_gpus * args.num_nodes
    config.TRAINER.TRUE_BATCH_SIZE = config.TRAINER.WORLD_SIZE * args.batch_size
    _scaling = config.TRAINER.TRUE_BATCH_SIZE / config.TRAINER.CANONICAL_BS
    config.TRAINER.SCALING = _scaling
    config.TRAINER.TRUE_LR = config.TRAINER.CANONICAL_LR * _scaling
    config.TRAINER.WARMUP_STEP = math.floor(config.TRAINER.WARMUP_STEP / _scaling)

    profiler = build_profiler(args.profiler_name)
    model = PL_JamMa(config, pretrained_ckpt=args.ckpt_path, profiler=profiler, dump_dir=args.dump_dir)
    loguru_logger.info(f"LoFTR LightningModule initialized!")

    data_module = MultiSceneDataModule(args, config)
    loguru_logger.info(f"LoFTR DataModule initialized!")

    logger = TensorBoardLogger(save_dir='jamma_log/', name=args.exp_name, default_hp_metric=False)
    ckpt_dir = Path(logger.log_dir) / 'checkpoints'

    ckpt_callback = ModelCheckpoint(monitor='auc@10', verbose=True, save_top_k=3, mode='max',
                                    save_last=True,
                                    dirpath=str(ckpt_dir),
                                    filename='{epoch}-{auc@5:.3f}-{auc@10:.3f}-{auc@20:.3f}')
    lr_monitor = LearningRateMonitor(logging_interval='step')
    callbacks = [lr_monitor]
    if not args.disable_ckpt:
        callbacks.append(ckpt_callback)

    trainer = pl.Trainer.from_argparse_args(
        args,
        strategy=DDPStrategy(find_unused_parameters=False),
        gradient_clip_val=config.TRAINER.GRADIENT_CLIPPING,
        callbacks=callbacks,
        logger=logger,
        replace_sampler_ddp=False,
        reload_dataloaders_every_n_epochs=0,
        enable_model_summary=True,
        profiler=profiler,)

    loguru_logger.info(f"Trainer initialized!")
    loguru_logger.info(f"Start training!")
    trainer.fit(model, datamodule=data_module)


if __name__ == '__main__':
    main()

In [ ]:
import numpy as np
import glob

npz_files = glob.glob('/kaggle/working/JamMa/src/datasets/RoadScene dataset/index/scene_info/*.npz')
d = np.load(npz_files[0], allow_pickle=True)
print(d['pair_infos'])
print(type(d['pair_infos'][0]))

In [ ]:
import numpy as np
import glob

npz_files = glob.glob('/kaggle/working/JamMa/src/datasets/RoadScene dataset/index/scene_info/*.npz')

for npz_path in npz_files:
    d = dict(np.load(npz_path, allow_pickle=True))
    
    new_pair_infos = np.array([((0, 1), 1.0, None)], dtype=object)
    d['pair_infos'] = new_pair_infos
    
    np.savez(npz_path, **d)

print(f"Fixed {len(npz_files)} npz files.")

# Verify one
d = np.load(npz_files[0], allow_pickle=True)
print(d['pair_infos'])
print(d['pair_infos'][0])

In [ ]:
# metrics.py
!sed -i 's/np\.trapz/np.trapezoid/g' /kaggle/working/JamMa/src/utils/metrics.py

# lightning_jamma.py
!sed -i 's/self\.trainer\.running_sanity_check/self.trainer.sanity_checking/' \
    /kaggle/working/JamMa/src/lightning/lightning_jamma.py

# megadepth.py
!sed -i "s/self\.scene_info = np\.load(\(.*\))/self.scene_info = dict(np.load(\1))/" \
    /kaggle/working/JamMa/src/datasets/megadepth.py

# data.py
!sed -i 's/self\.world_size = dist\.get_world_size()/self.world_size = dist.get_world_size() if dist.is_initialized() else 1/' \
    /kaggle/working/JamMa/src/lightning/data.py
!sed -i 's/self\.rank = dist\.get_rank()/self.rank = dist.get_rank() if dist.is_initialized() else 0/' \
    /kaggle/working/JamMa/src/lightning/data.py
!sed -i 's/sampler = DistributedSampler(self\.val_dataset, shuffle=False)/sampler = DistributedSampler(self.val_dataset, shuffle=False) if dist.is_initialized() else SequentialSampler(self.val_dataset)/' \
    /kaggle/working/JamMa/src/lightning/data.py
!sed -i 's/sampler = DistributedSampler(dataset, shuffle=False)/sampler = DistributedSampler(dataset, shuffle=False) if dist.is_initialized() else SequentialSampler(dataset)/' \
    /kaggle/working/JamMa/src/lightning/data.py
!sed -i 's/sampler = DistributedSampler(self\.test_dataset, shuffle=False)/sampler = DistributedSampler(self.test_dataset, shuffle=False) if dist.is_initialized() else SequentialSampler(self.test_dataset)/' \
    /kaggle/working/JamMa/src/lightning/data.py
!sed -i '1s/^/from torch.utils.data import SequentialSampler\n/' \
    /kaggle/working/JamMa/src/lightning/data.py

# data config path
!sed -i 's|/path/to/data/roadscene|/kaggle/working/JamMa/src/datasets/RoadScene dataset|g' \
    /kaggle/working/JamMa/configs/data/roadscene_trainval.py


In [ ]:
import numpy as np, glob

npz_files = glob.glob('/kaggle/working/JamMa/src/datasets/RoadScene dataset/index/scene_info/*.npz')
d = np.load(npz_files[0], allow_pickle=True)
print("depth_paths:", d['depth_paths'])
print("pair_infos:", d['pair_infos'])

In [ ]:
import numpy as np, h5py, glob
from PIL import Image
import os

dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
npz_files = glob.glob(f'{dataset_root}/index/scene_info/*.npz')
print(f"Found {len(npz_files)} npz files")

for split in ['train', 'test']:
    os.makedirs(os.path.join(dataset_root, split, 'depths'), exist_ok=True)

def find_image_split(dataset_root, img_rel_path):
    for split in ['train', 'test']:
        candidate = os.path.join(dataset_root, split, img_rel_path)
        if os.path.exists(candidate):
            return split, candidate
    raise FileNotFoundError(f"Cannot find {img_rel_path}")

for npz_path in npz_files:
    d = dict(np.load(npz_path, allow_pickle=True))

    d['pair_infos'] = np.array([((0, 1), 1.0, None)], dtype=object)

    image_paths = d['image_paths']
    new_depth_paths = []
    for img_rel_path in image_paths:
        split, img_full_path = find_image_split(dataset_root, img_rel_path)
        with Image.open(img_full_path) as img:
            w, h = img.size
        stem = os.path.splitext(os.path.basename(img_rel_path))[0]
        modality = 'vis' if 'visible' in img_rel_path else 'ir'
        depth_filename = f"{stem}_{modality}.h5"
        depth_rel_path = f"depths/{depth_filename}"
        depth_full_path = os.path.join(dataset_root, split, depth_rel_path)
        if not os.path.exists(depth_full_path):
            with h5py.File(depth_full_path, 'w') as f:
                f.create_dataset('depth', data=np.full((h, w), 100.0, dtype=np.float32))
        new_depth_paths.append(depth_rel_path)

    d['depth_paths'] = np.array(new_depth_paths)
    np.savez(npz_path, **d)

print("Done! Verifying...")
d = np.load(npz_files[0], allow_pickle=True)
print("depth_paths:", d['depth_paths'])
print("pair_infos:", d['pair_infos'])

In [ ]:
import cv2, os

dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
for split in ['train', 'test']:
    vis_dir = f'{dataset_root}/{split}/visible'
    ir_dir  = f'{dataset_root}/{split}/infrared'
    for stem in os.listdir(vis_dir):
        vis = cv2.imread(f'{vis_dir}/{stem}')
        ir  = cv2.imread(f'{ir_dir}/{stem}')
        if vis is None or ir is None: continue
        H, W = vis.shape[:2]
        if ir.shape[:2] != (H, W):
            cv2.imwrite(f'{ir_dir}/{stem}', cv2.resize(ir, (W, H)))
print("Done resizing IR images.")

In [ ]:
%%writefile /kaggle/working/JamMa/configs/jamma/outdoor/finetune_roadscene.py

import subprocess
import sys
import os

for pkg, mod in [('mamba-ssm', 'mamba_ssm'), ('causal-conv1d', 'causal_conv1d'),
                  ('yacs', 'yacs'), ('kornia', 'kornia'), ('einops', 'einops'),
                  ('loguru', 'loguru')]:
    try:
        __import__(mod)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

from kornia.utils import create_meshgrid
from einops.einops import rearrange
from loguru import logger

SCRIPT_DIR = Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPT_DIR))

import types as _types
try:
    import pytorch_lightning.profiler  # noqa: F401
except (ImportError, ModuleNotFoundError):
    try:
        from pytorch_lightning.profilers import SimpleProfiler, PassThroughProfiler
        _mock = _types.ModuleType('pytorch_lightning.profiler')
        _mock.SimpleProfiler = SimpleProfiler
        _mock.PassThroughProfiler = PassThroughProfiler
        sys.modules['pytorch_lightning.profiler'] = _mock
    except ImportError:
        pass

if torch.cuda.is_available():
    _cap = torch.cuda.get_device_capability()
    if _cap[0] < 7:
        raise RuntimeError(
            f'GPU compute capability {_cap[0]}.{_cap[1]} is not supported by mamba-ssm. '
            'Switch to a T4 or newer GPU (sm_70+) in Kaggle: Settings > Accelerator > GPU T4 x2.'
        )

from src.jamma.jamma import JamMa
from src.jamma.backbone import CovNextV2_nano
from src.config.default import get_cfg_defaults
from src.utils.misc import lower_config

ROADSCENE_DIR  = SCRIPT_DIR.parent / 'RoadScene'
VIS_DIR        = ROADSCENE_DIR / 'crop_HR_visible'
IR_DIR         = ROADSCENE_DIR / 'cropinfrared'
IMG_W, IMG_H   = 320, 256
BATCH_SIZE     = 2
NUM_EPOCHS     = 30
LR             = 5e-5
DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR       = SCRIPT_DIR / 'finetune_output'
VIZ_SAMPLES    = 4
MAX_SHIFT      = 0.12
SUB_W          = 0.5
JAMMA_URL      = 'https://github.com/leoluxxx/JamMa/releases/download/v0.1/jamma.ckpt'

SAVE_DIR.mkdir(exist_ok=True)
(SAVE_DIR / 'checkpoints').mkdir(exist_ok=True)
(SAVE_DIR / 'viz').mkdir(exist_ok=True)


class RoadSceneDataset(Dataset):
    def __init__(self, vis_dir, ir_dir, img_w=IMG_W, img_h=IMG_H,
                 max_shift=MAX_SHIFT, augment=True):
        self.img_w, self.img_h = img_w, img_h
        self.max_shift = max_shift
        self.augment = augment

        vis_dir, ir_dir = Path(vis_dir), Path(ir_dir)
        vs = {p.stem for p in vis_dir.glob('*.jpg')}
        irs = {p.stem for p in ir_dir.glob('*.jpg')}
        stems = sorted(vs & irs)
        self.pairs = [(vis_dir / f'{s}.jpg', ir_dir / f'{s}.jpg') for s in stems]
        logger.info(f'Dataset: {len(self.pairs)} visible-IR pairs')

    def _load(self, path, gray=False):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE if gray else cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(str(path))
        img = cv2.resize(img, (self.img_w, self.img_h), interpolation=cv2.INTER_LINEAR)
        if gray:
            img = np.stack([img] * 3, axis=-1)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img

    def _rand_H(self):
        H, W = self.img_h, self.img_w
        src = np.float32([[0,0],[W,0],[W,H],[0,H]])
        noise = np.random.uniform(-1, 1, (4, 2)).astype(np.float32)
        dst = src + noise * np.float32([self.max_shift * W, self.max_shift * H])
        return cv2.getPerspectiveTransform(src, dst)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        vp, ip = self.pairs[idx]
        vis = self._load(vp, gray=False)
        ir  = self._load(ip, gray=True)

        H0 = self._rand_H() if self.augment else np.eye(3, dtype=np.float32)
        H1 = self._rand_H() if self.augment else np.eye(3, dtype=np.float32)

        kw = dict(dsize=(self.img_w, self.img_h), flags=cv2.INTER_LINEAR,
                  borderMode=cv2.BORDER_CONSTANT)
        vis_w = cv2.warpPerspective(vis, H0, **kw)
        ir_w  = cv2.warpPerspective(ir,  H1, **kw)

        H_rel = H1 @ np.linalg.inv(H0)

        img0 = torch.from_numpy(vis_w).float().permute(2,0,1) / 255.0
        img1 = torch.from_numpy(ir_w).float().permute(2,0,1) / 255.0
        Ht   = torch.from_numpy(H_rel.astype(np.float32))

        return {
            'imagec_0':  img0,
            'imagec_1':  img1,
            'H_0to1':    Ht,
            'pair_names': (vp.name, ip.name),
        }


def collate_fn(batch):
    out = {}
    for k in batch[0]:
        if k == 'pair_names':
            out[k] = tuple(zip(*[b[k] for b in batch]))
        else:
            out[k] = torch.stack([b[k] for b in batch])
    return out


@torch.no_grad()
def _warp_H(kpts, H, h_tgt, w_tgt):
    N, L, _ = kpts.shape
    ones = torch.ones(N, L, 1, device=kpts.device, dtype=kpts.dtype)
    kh   = torch.cat([kpts, ones], -1)
    wh   = (H @ kh.transpose(1,2)).transpose(1,2)
    w    = wh[..., :2] / (wh[..., [2]] + 1e-6)
    valid = (w[...,0] > 0) & (w[...,0] < w_tgt-1) & \
            (w[...,1] > 0) & (w[...,1] < h_tgt-1)
    return valid, w


@torch.no_grad()
def compute_supervision_coarse_h(data, config):
    dev = data['imagec_0'].device
    N, _, H0, W0 = data['imagec_0'].shape
    _, _, H1, W1 = data['imagec_1'].shape
    sc = config['jamma']['resolution'][0]
    h0, w0 = H0//sc, W0//sc
    h1, w1 = H1//sc, W1//sc

    Hm  = data['H_0to1'].to(dev)
    Him = torch.inverse(Hm)

    g0 = create_meshgrid(h0, w0, False, dev).reshape(1, h0*w0, 2).repeat(N,1,1)
    g1 = create_meshgrid(h1, w1, False, dev).reshape(1, h1*w1, 2).repeat(N,1,1)
    p0 = sc * g0
    p1 = sc * g1

    v0, wp0 = _warp_H(p0, Hm,  H1, W1)
    v1, wp1 = _warp_H(p1, Him, H0, W0)
    wp0[~v0] = 0
    wp1[~v1] = 0

    c0 = wp0 / sc
    c1 = wp1 / sc

    def oob(pt, w, h):
        return (pt[...,0]<0)|(pt[...,0]>=w)|(pt[...,1]<0)|(pt[...,1]>=h)

    r0 = c0.round().long()
    r1 = c1.round().long()
    ni1 = r0[...,0] + r0[...,1]*w1
    ni0 = r1[...,0] + r1[...,1]*w0
    ni1[oob(r0,w1,h1)|~v0] = 0
    ni0[oob(r1,w0,h0)|~v1] = 0

    a1 = torch.arange(h0*w0, device=dev)[None].repeat(N,1)
    a0 = torch.arange(h1*w1, device=dev)[None].repeat(N,1)
    a1[ni1==0] = 0
    a0[ni0==0] = 0
    ab = torch.arange(N, device=dev).unsqueeze(1)

    cm = torch.zeros(N, h0*w0, h1*w1, device=dev)
    cm[ab, a1, ni1] = 1
    cm[ab, ni0, a0] = 1
    cm[:, 0, 0] = False

    b_ids, i_ids, j_ids = cm.nonzero(as_tuple=True)
    if len(b_ids) == 0:
        b_ids = torch.tensor([0], device=dev)
        i_ids = torch.tensor([0], device=dev)
        j_ids = torch.tensor([0], device=dev)

    data.update({
        'conf_matrix_gt':   cm,
        'spv_b_ids':        b_ids,
        'spv_i_ids':        i_ids,
        'spv_j_ids':        j_ids,
        'num_candidates_max': int(b_ids.shape[0]),
        'spv_w_pt0_i':      wp0,
        'spv_pt1_i':        p1,
        'dataset_name':     ['roadscene'] * N,
    })


@torch.no_grad()
def compute_supervision_fine_h(data, config):
    W_f  = config['jamma']['fine_window_size']
    sc_c = config['jamma']['resolution'][0]
    sc_f = config['jamma']['resolution'][1]
    sfc  = sc_c // sc_f

    dev = data['imagec_0'].device
    N, _, H0, W0 = data['imagec_0'].shape
    _, _, H1, W1 = data['imagec_1'].shape
    h0f, w0f = H0//sc_f, W0//sc_f
    h1f, w1f = H1//sc_f, W1//sc_f

    b_ids = data['b_ids_fine']
    i_ids = data['i_ids_fine']
    j_ids = data['j_ids_fine']

    if len(b_ids) == 0:
        data.update({'conf_matrix_f_gt': torch.zeros(1,W_f**2,W_f**2,device=dev)})
        return

    Hm  = data['H_0to1'].to(dev)
    Him = torch.inverse(Hm)
    stride_f = data['hw0_f'][0] // data['hw0_c'][0]
    pad = 0 if W_f % 2 == 0 else W_f // 2

    def make_window_grid(hf, wf, n, sc):
        g = create_meshgrid(hf, wf, False, dev).repeat(n,1,1,1) * sc
        g = rearrange(g, 'n h w c -> n c h w')
        g = F.unfold(g, kernel_size=(W_f,W_f), stride=stride_f, padding=pad)
        return rearrange(g, 'n (c ww) l -> n l ww c', ww=W_f**2)

    g0 = make_window_grid(h0f, w0f, N, sc_f)[b_ids, i_ids]  # [M, W^2, 2]
    g1 = make_window_grid(h1f, w1f, N, sc_f)[b_ids, j_ids]

    M  = b_ids.shape[0]
    Hbm  = Hm[b_ids]
    Hibm = Him[b_ids]

    def batch_warp(pts, H):
        ph = torch.cat([pts, torch.ones(M,W_f**2,1,device=dev)],-1)
        wh = (H @ ph.transpose(1,2)).transpose(1,2)
        return wh[...,:2] / (wh[...,[2]] + 1e-6)

    wp0 = batch_warp(g0, Hbm)
    wp1 = batch_warp(g1, Hibm)

    c0f = torch.stack([i_ids % data['hw0_c'][1],
                       i_ids // data['hw0_c'][1]], dim=1).float() * sfc - pad
    c1f = torch.stack([j_ids % data['hw1_c'][1],
                       j_ids // data['hw1_c'][1]], dim=1).float() * sfc - pad

    w0f_rel = wp0 / sc_f - c1f[:,None,:]
    w1f_rel = wp1 / sc_f - c0f[:,None,:]

    def oob(pt, w, h):
        return (pt[...,0]<0)|(pt[...,0]>=w)|(pt[...,1]<0)|(pt[...,1]>=h)

    r0 = w0f_rel.round().long()
    r1 = w1f_rel.round().long()
    ni1 = r0[...,0] + r0[...,1]*W_f
    ni0 = r1[...,0] + r1[...,1]*W_f
    ni1[oob(r0,W_f,W_f)] = 0
    ni0[oob(r1,W_f,W_f)] = 0

    loop = torch.stack([ni0[b][ni1[b]] for b in range(M)], dim=0)
    ref  = torch.arange(W_f**2, device=dev)[None].repeat(M,1)
    ok   = (loop == ref)
    ok[:,0] = False

    cmf = torch.zeros(M, W_f**2, W_f**2, device=dev)
    bi, ii = torch.where(ok)
    ji = ni1[bi, ii]
    cmf[bi, ii, ji] = 1
    data.update({'conf_matrix_f_gt': cmf})


class CustomLoss(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.lc = config['jamma']['loss']

    def _focal(self, pred, gt, pos_w, neg_w=None):
        nw  = neg_w if neg_w is not None else self.lc['pos_weight']
        alpha, gamma = self.lc['focal_alpha'], self.lc['focal_gamma']
        pm, nm = gt > 0, gt == 0
        pw_use = pos_w
        if not pm.any():
            pm[0,0,0] = True; pw_use = 0.
        if not nm.any():
            nm[0,0,0] = True; nw = 0.
        pred = pred.clamp(1e-6, 1-1e-6)
        lp = -alpha*(1-pred[pm]).pow(gamma)*pred[pm].log()
        ln = -alpha*pred[nm].pow(gamma)*(1-pred[nm]).log()
        return pw_use*lp.mean() + nw*ln.mean()

    def forward(self, data):
        pw = self.lc['pos_weight']
        c01 = data['conf_matrix_0_to_1'].clamp(1e-6,1-1e-6)
        c10 = data['conf_matrix_1_to_0'].clamp(1e-6,1-1e-6)
        gt  = data['conf_matrix_gt']
        pm  = gt == 1
        pw_c = pw
        if not pm.any():
            pm[0,0,0] = True; pw_c = 0.
        alpha, gamma = self.lc['focal_alpha'], self.lc['focal_gamma']
        lp = -alpha*(1-c01[pm]).pow(gamma)*c01[pm].log()
        lp += -alpha*(1-c10[pm]).pow(gamma)*c10[pm].log()
        lc = pw_c * lp.mean() * self.lc['coarse_weight']

        lf = self._focal(data['conf_matrix_fine'], data['conf_matrix_f_gt'], pw) \
             * self.lc['fine_weight']

        loss = lc + lf
        scalars = {'loss_c': lc.detach().cpu(), 'loss_f': lf.detach().cpu()}

        m_bids = data.get('m_bids')
        pts0   = data.get('mkpts0_f_train')
        pts1   = data.get('mkpts1_f_train')
        if m_bids is not None and pts0 is not None and len(pts0) > 0:
            Hpm  = data['H_0to1'][m_bids]
            ph   = torch.cat([pts0, torch.ones(len(pts0),1,device=pts0.device)],-1)
            wh   = (Hpm @ ph.unsqueeze(-1)).squeeze(-1)
            wp0  = wh[:,:2] / (wh[:,[2]] + 1e-6)
            dist = (wp0 - pts1).norm(dim=-1)
            ok   = dist < 4.0
            ls   = (dist[ok].mean() if ok.any() else dist.mean() * 1e-9) * SUB_W
            loss = loss + ls
            scalars['loss_sub'] = ls.detach().cpu()

        scalars['loss'] = loss.detach().cpu()
        data.update({'loss': loss, 'loss_scalars': scalars})


def load_pretrained(backbone, matcher, device):
    logger.info('Downloading JamMa pretrained weights...')
    state = torch.hub.load_state_dict_from_url(JAMMA_URL, file_name='jamma.ckpt',
                                               map_location='cpu')['state_dict']
    bk = {k[9:]: v for k, v in state.items() if k.startswith('backbone.')}
    mt = {k[8:]: v for k, v in state.items() if k.startswith('matcher.')}
    backbone.load_state_dict(bk, strict=True)
    matcher.load_state_dict(mt, strict=True)
    logger.info('Pretrained weights loaded.')


def tensor_to_np(t):
    img = t.permute(1,2,0).cpu().numpy()
    return (img * 255).clip(0,255).astype(np.uint8)


def draw_matches(img0, img1, kp0, kp1, conf, max_kp=150):
    H, W = img0.shape[:2]
    gap = 4
    canvas = np.full((H, 2*W+gap, 3), 30, dtype=np.uint8)
    canvas[:, :W]      = img0
    canvas[:, W+gap:]  = img1
    if len(kp0) == 0:
        return canvas
    idx = np.argsort(-conf)[:max_kp]
    cmap = plt.cm.plasma
    for i in idx:
        c_val = float(conf[i])
        c_rgb = tuple(int(x*255) for x in cmap(c_val)[:3])
        x0, y0 = int(kp0[i,0]), int(kp0[i,1])
        x1, y1 = int(kp1[i,0])+W+gap, int(kp1[i,1])
        cv2.line(canvas,(x0,y0),(x1,y1),c_rgb,1,cv2.LINE_AA)
        cv2.circle(canvas,(x0,y0),3,c_rgb,-1)
        cv2.circle(canvas,(x1,y1),3,c_rgb,-1)
    return canvas


@torch.no_grad()
def run_inference(backbone, matcher, batch, device):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(device)
    backbone(batch)
    matcher(batch, mode='test')
    kp0 = batch.get('mkpts0_f', batch.get('mkpts0_c', torch.zeros(0,2))).cpu().numpy()
    kp1 = batch.get('mkpts1_f', batch.get('mkpts1_c', torch.zeros(0,2))).cpu().numpy()
    cf  = batch.get('mconf_f', batch.get('mconf', torch.zeros(len(kp0)))).cpu().numpy()
    return kp0, kp1, cf


def compute_transfer_errors(kp0, kp1, H_np):
    if len(kp0) == 0:
        return np.array([])
    kp0h = np.concatenate([kp0, np.ones((len(kp0),1))], axis=1)
    wkp0 = (H_np @ kp0h.T).T
    wkp0 = wkp0[:,:2] / (wkp0[:,[2]] + 1e-6)
    return np.linalg.norm(wkp0 - kp1, axis=1)


def visualize_epoch(backbone, matcher, dataset, epoch, config, device, loss_history):
    backbone.eval(); matcher.eval()
    n = min(VIZ_SAMPLES, len(dataset))
    idxs = np.linspace(0, len(dataset)-1, n, dtype=int)

    fig = plt.figure(figsize=(22, 6*n))
    gs_outer = gridspec.GridSpec(n, 1, figure=fig, hspace=0.45)

    all_errors = []
    all_n_matches = []

    for row, idx in enumerate(idxs):
        sample = dataset[idx]
        batch = collate_fn([sample])
        for k,v in batch.items():
            if isinstance(v, torch.Tensor): batch[k] = v.to(device)

        kp0, kp1, cf = run_inference(backbone, matcher, batch, device)
        H_np = batch['H_0to1'][0].cpu().numpy()
        errs = compute_transfer_errors(kp0, kp1, H_np)
        all_errors.extend(errs.tolist())
        all_n_matches.append(len(kp0))

        img0 = tensor_to_np(batch['imagec_0'][0].cpu())
        img1 = tensor_to_np(batch['imagec_1'][0].cpu())
        canvas = draw_matches(img0, img1, kp0, kp1, cf)

        gs_inner = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_outer[row],
                                                    wspace=0.08)

        ax0 = fig.add_subplot(gs_inner[0, :2])
        ax0.imshow(canvas[:,:,::-1])
        n_inliers = int((errs < 2.0).sum()) if len(errs) > 0 else 0
        ax0.set_title(f'Pair {idx} | {len(kp0)} matches | {n_inliers} inliers (<2px) | '
                      f'median err {np.median(errs):.2f}px' if len(errs)>0 else
                      f'Pair {idx} | 0 matches', fontsize=10)
        ax0.axis('off')

        ax1 = fig.add_subplot(gs_inner[0, 2])
        if len(errs) > 0:
            ax1.hist(errs.clip(0, 30), bins=40, color='steelblue', edgecolor='k', linewidth=0.4)
            ax1.axvline(2.0, color='red', linestyle='--', linewidth=1.2, label='2px')
            ax1.axvline(np.median(errs), color='orange', linestyle='--', linewidth=1.2,
                        label=f'median={np.median(errs):.1f}px')
            ax1.legend(fontsize=8)
        ax1.set_xlabel('Transfer error (px)', fontsize=9)
        ax1.set_ylabel('Count', fontsize=9)
        ax1.set_title('Error distribution', fontsize=10)

    viz_dir = SAVE_DIR / 'viz' / f'epoch_{epoch:03d}'
    viz_dir.mkdir(parents=True, exist_ok=True)
    fig.suptitle(f'Epoch {epoch} — JamMa fine-tuned on RoadScene (visible↔IR)', fontsize=13)
    fig.savefig(viz_dir / 'matches.png', bbox_inches='tight', dpi=110)
    plt.close(fig)

    if loss_history:
        plot_loss_curves(loss_history, viz_dir / 'loss.png')

    backbone.train(); matcher.train()
    return all_errors, all_n_matches


def plot_loss_curves(history, path):
    keys = ['loss', 'loss_c', 'loss_f', 'loss_sub']
    colors = ['black', 'steelblue', 'coral', 'mediumseagreen']
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    for k, c in zip(keys, colors):
        if k in history and len(history[k]) > 0:
            vals = history[k]
            ax.plot(vals, label=k, color=c, linewidth=1.8)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title('Training Losses'); ax.legend(); ax.grid(alpha=0.3)

    ax2 = axes[1]
    if 'n_matches' in history and len(history['n_matches']) > 0:
        ax2.plot(history['n_matches'], color='purple', linewidth=1.8, label='avg matches/pair')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('# matches')
        ax2.set_title('Matches per pair'); ax2.legend(); ax2.grid(alpha=0.3)
    if 'median_err' in history and len(history['median_err']) > 0:
        ax3 = ax2.twinx()
        ax3.plot(history['median_err'], color='darkorange', linewidth=1.8,
                 linestyle='--', label='median err (px)')
        ax3.set_ylabel('Median transfer error (px)', color='darkorange')
        ax3.legend(loc='upper right')

    fig.tight_layout()
    fig.savefig(path, dpi=110, bbox_inches='tight')
    plt.close(fig)


def visualize_before_after(backbone, matcher, dataset, before_state, config, device):
    n = min(4, len(dataset))
    idxs = np.linspace(0, len(dataset)-1, n, dtype=int)

    fig, axes = plt.subplots(n, 2, figsize=(20, 5*n))
    fig.suptitle('Before (pretrained) vs After (fine-tuned) on visible↔IR', fontsize=13)

    for row, idx in enumerate(idxs):
        sample = dataset[idx]
        batch_after = collate_fn([sample])
        for k,v in batch_after.items():
            if isinstance(v, torch.Tensor): batch_after[k] = v.to(device)

        batch_before = {k: v.clone() if isinstance(v, torch.Tensor) else v
                        for k, v in batch_after.items()}
        batch_before['imagec_0'] = batch_after['imagec_0'].clone()
        batch_before['imagec_1'] = batch_after['imagec_1'].clone()

        backbone.load_state_dict(before_state['backbone'])
        matcher.load_state_dict(before_state['matcher'])
        backbone.eval(); matcher.eval()
        kp0_b, kp1_b, cf_b = run_inference(backbone, matcher, batch_before, device)

        backbone.load_state_dict(before_state['_after_backbone'])
        matcher.load_state_dict(before_state['_after_matcher'])
        backbone.eval(); matcher.eval()
        kp0_a, kp1_a, cf_a = run_inference(backbone, matcher, batch_after, device)

        H_np = batch_after['H_0to1'][0].cpu().numpy()
        img0 = tensor_to_np(batch_after['imagec_0'][0].cpu())
        img1 = tensor_to_np(batch_after['imagec_1'][0].cpu())

        err_b = compute_transfer_errors(kp0_b, kp1_b, H_np)
        err_a = compute_transfer_errors(kp0_a, kp1_a, H_np)

        cb = draw_matches(img0, img1, kp0_b, kp1_b, cf_b)
        ca = draw_matches(img0, img1, kp0_a, kp1_a, cf_a)

        for col, (canvas, errs, tag) in enumerate([
            (cb, err_b, f'Pretrained | {len(kp0_b)} matches | med={np.median(errs):.1f}px' if len(errs)>0 else f'Pretrained | 0 matches'),
            (ca, err_a, f'Fine-tuned  | {len(kp0_a)} matches | med={np.median(errs):.1f}px' if len(errs)>0 else f'Fine-tuned  | 0 matches'),
        ]):
            ax = axes[row, col] if n > 1 else axes[col]
            ax.imshow(canvas[:,:,::-1])
            ax.set_title(tag, fontsize=10)
            ax.axis('off')

    fig.tight_layout()
    fig.savefig(SAVE_DIR / 'viz' / 'before_after.png', dpi=110, bbox_inches='tight')
    plt.close(fig)


def train_step(batch, backbone, matcher, loss_fn, config, device):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            batch[k] = v.to(device)

    compute_supervision_coarse_h(batch, config)
    backbone(batch)
    matcher(batch, mode='train')
    compute_supervision_fine_h(batch, config)
    loss_fn(batch)
    return batch['loss'], batch['loss_scalars']


def main():
    torch.manual_seed(42)
    np.random.seed(42)

    cfg = get_cfg_defaults()
    config = lower_config(cfg)

    logger.info(f'Device: {DEVICE}')
    logger.info('Building model...')

    backbone = CovNextV2_nano().to(DEVICE)
    matcher  = JamMa(config=config['jamma']).to(DEVICE)

    load_pretrained(backbone, matcher, DEVICE)

    before_bk_state = {k: v.cpu().clone() for k, v in backbone.state_dict().items()}
    before_mt_state = {k: v.cpu().clone() for k, v in matcher.state_dict().items()}

    loss_fn = CustomLoss(config).to(DEVICE)

    dataset   = RoadSceneDataset(VIS_DIR, IR_DIR, augment=True)
    val_dset  = RoadSceneDataset(VIS_DIR, IR_DIR, augment=False)
    loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, collate_fn=collate_fn, drop_last=True)

    for p in backbone.parameters():
        p.requires_grad_(False)
    for p in matcher.parameters():
        p.requires_grad_(True)

    optimizer = torch.optim.AdamW(
        [{'params': matcher.parameters(), 'lr': LR},
         {'params': backbone.parameters(), 'lr': LR * 0.1}],
        weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LR*0.05)

    history = {k: [] for k in ['loss','loss_c','loss_f','loss_sub','n_matches','median_err']}

    logger.info('Starting fine-tuning...')

    for epoch in range(1, NUM_EPOCHS + 1):
        if epoch == 6:
            for p in backbone.parameters():
                p.requires_grad_(True)
            logger.info('Backbone unfrozen at epoch 6.')

        backbone.train(); matcher.train()
        ep_scalars = {k: [] for k in ['loss','loss_c','loss_f','loss_sub']}
        t0 = time.time()

        for step, batch in enumerate(loader):
            optimizer.zero_grad()
            try:
                loss, scalars = train_step(batch, backbone, matcher, loss_fn, config, DEVICE)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(backbone.parameters()) + list(matcher.parameters()), 0.5)
                optimizer.step()
                for k in ep_scalars:
                    if k in scalars:
                        ep_scalars[k].append(float(scalars[k]))
            except Exception as e:
                logger.warning(f'Step {step} failed: {e}')
                continue

        scheduler.step()

        for k in ['loss','loss_c','loss_f','loss_sub']:
            v = ep_scalars[k]
            history[k].append(float(np.mean(v)) if v else 0.0)

        errs, nm = visualize_epoch(backbone, matcher, val_dset, epoch, config, DEVICE, history)
        history['n_matches'].append(float(np.mean(nm)) if nm else 0.0)
        history['median_err'].append(float(np.median(errs)) if errs else 0.0)

        elapsed = time.time() - t0
        logger.info(
            f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
            f'loss={history["loss"][-1]:.4f} | '
            f'loss_c={history["loss_c"][-1]:.4f} | '
            f'loss_f={history["loss_f"][-1]:.4f} | '
            f'matches={history["n_matches"][-1]:.0f} | '
            f'med_err={history["median_err"][-1]:.2f}px | '
            f'{elapsed:.0f}s'
        )

        ckpt = {
            'epoch': epoch,
            'backbone': backbone.state_dict(),
            'matcher':  matcher.state_dict(),
            'optimizer': optimizer.state_dict(),
            'history':  history,
        }
        torch.save(ckpt, SAVE_DIR / 'checkpoints' / f'epoch_{epoch:03d}.pt')
        torch.save(ckpt, SAVE_DIR / 'checkpoints' / 'latest.pt')

    plot_loss_curves(history, SAVE_DIR / 'viz' / 'final_loss_curves.png')

    before_state = {
        'backbone': before_bk_state,
        'matcher':  before_mt_state,
        '_after_backbone': {k: v.cpu().clone() for k, v in backbone.state_dict().items()},
        '_after_matcher':  {k: v.cpu().clone() for k, v in matcher.state_dict().items()},
    }
    visualize_before_after(backbone, matcher, val_dset, before_state, config, DEVICE)

    visualize_final_summary(history)
    logger.info(f'Done. Outputs saved to {SAVE_DIR}')


def visualize_final_summary(history):
    fig = plt.figure(figsize=(18, 10))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    ax = fig.add_subplot(gs[0, 0])
    epochs = range(1, len(history['loss'])+1)
    ax.plot(epochs, history['loss'],   'k-',  lw=2, label='total')
    ax.plot(epochs, history['loss_c'], 'b-',  lw=1.5, label='coarse')
    ax.plot(epochs, history['loss_f'], 'r-',  lw=1.5, label='fine')
    if any(v > 0 for v in history['loss_sub']):
        ax.plot(epochs, history['loss_sub'], 'g--', lw=1.5, label='sub-px')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Loss components')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(epochs, history['n_matches'], 'purple', lw=2)
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Avg matches / pair')
    ax2.set_title('Matches per pair'); ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[0, 2])
    ax3.plot(epochs, history['median_err'], 'darkorange', lw=2)
    ax3.set_xlabel('Epoch'); ax3.set_ylabel('Median transfer error (px)')
    ax3.set_title('Match accuracy (lower = better)'); ax3.grid(alpha=0.3)

    ax4 = fig.add_subplot(gs[1, :])
    if history['median_err']:
        best_epoch = int(np.argmin(history['median_err'])) + 1
        ax4.bar(list(epochs), history['median_err'], color='steelblue', alpha=0.7)
        ax4.axvline(best_epoch, color='red', linestyle='--', lw=2,
                    label=f'Best epoch: {best_epoch}')
        ax4.set_xlabel('Epoch'); ax4.set_ylabel('Median transfer error (px)')
        ax4.set_title('Transfer error per epoch'); ax4.legend(); ax4.grid(alpha=0.3, axis='y')

    fig.suptitle('JamMa Fine-tuning on RoadScene (visible↔IR) — Summary', fontsize=14)
    fig.savefig(SAVE_DIR / 'viz' / 'summary.png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    logger.info(f'Summary saved to {SAVE_DIR}/viz/summary.png')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/JamMa/configs/jamma/outdoor/finetune_roadscene.py
from configs.jamma.outdoor.final import cfg

cfg.TRAINER.CANONICAL_LR = 1e-4   # reduced from default for fine-tuning
cfg.TRAINER.WARMUP_STEP = 58200     # shorter warmup for fine-tuning

In [ ]:
!python -W ignore /kaggle/working/JamMa/train.py \
    /kaggle/working/JamMa/configs/data/roadscene_trainval.py \
    /kaggle/working/JamMa/configs/jamma/outdoor/final.py \
    --exp_name roadscene_finetune \
    --ckpt_path /kaggle/working/JamMa/weights/jamma.ckpt \
    --gpus 1 --batch_size 2 --max_epochs 50

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/JamMa')

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from src.config.default import get_cfg_defaults
from src.lightning.lightning_jamma import PL_JamMa

config = get_cfg_defaults()
config.merge_from_file('/kaggle/working/JamMa/configs/jamma/outdoor/finetune_roadscene.py')

model_pretrained = PL_JamMa(config,
    pretrained_ckpt='/kaggle/working/JamMa/weights/jamma.ckpt')

# Find best checkpoint from v2 training
import glob
ckpts = sorted(glob.glob('/kaggle/working/jamma_log/roadscene_finetune/version_*/checkpoints/epoch=*.ckpt'))
print("Available checkpoints:")
for c in ckpts:
    print(" ", c)
best_ckpt = ckpts[-1] if ckpts else '/kaggle/working/jamma_log/roadscene_finetune/version_0/checkpoints/last.ckpt'
print(f"\nUsing: {best_ckpt}")

model_finetuned = PL_JamMa(config, pretrained_ckpt=best_ckpt)

def get_target_size(path, target_size=640, df=16):
    img = cv2.imread(path)
    H, W = img.shape[:2]
    scale = target_size / max(H, W)
    return int(round(H * scale / df)) * df, int(round(W * scale / df)) * df

def load_image_for_jamma(path, target_H, target_W):
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    img_resized = cv2.resize(img_rgb, (target_W, target_H))
    tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float()[None] / 255.
    return tensor, (target_W / W, target_H / H), img_rgb

def run_inference(model, vis_path, ir_path):
    target_H, target_W = get_target_size(vis_path)
    img0, scale0, vis_orig = load_image_for_jamma(vis_path, target_H, target_W)
    img1, scale1, ir_orig  = load_image_for_jamma(ir_path,  target_H, target_W)
    H0, W0 = img0.shape[2], img0.shape[3]
    batch = {
        'imagec_0': img0.cuda(), 'imagec_1': img1.cuda(),
        'h_8': H0 // 8, 'w_8': W0 // 8,
        'dataset_name': ['RoadScene'], 'pair_names': [('vis', 'ir')],
    }
    with torch.no_grad():
        with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
            model.backbone(batch)
        model.matcher(batch, mode='test')
    return (batch['mkpts0_f'].cpu().numpy(),
            batch['mkpts1_f'].cpu().numpy(),
            target_H, target_W, vis_orig, ir_orig)

def evaluate_model(model, model_name):
    model.eval().cuda()
    dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
    test_images  = sorted(os.listdir(f'{dataset_root}/test/visible'))
    all_n, all_err, all_5, all_10 = [], [], [], []

    for stem in test_images:
        vis_path = f'{dataset_root}/test/visible/{stem}'
        ir_path  = f'{dataset_root}/test/infrared/{stem}'
        try:
            mkpts0, mkpts1, *_ = run_inference(model, vis_path, ir_path)
        except Exception:
            all_n.append(0); all_err.append(float('nan'))
            all_5.append(0); all_10.append(0)
            continue

        if len(mkpts0) > 0:
            reproj = np.linalg.norm(mkpts0 - mkpts1, axis=1)
            all_err.append(reproj.mean())
            all_5.append((reproj < 5).mean() * 100)
            all_10.append((reproj < 10).mean() * 100)
        else:
            all_err.append(float('nan')); all_5.append(0); all_10.append(0)
        all_n.append(len(mkpts0))

    valid_err = [e for e in all_err if not np.isnan(e)]
    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Avg matches per pair:        {np.mean(all_n):.1f}")
    print(f"  Pairs with >0 matches:       {sum(n>0 for n in all_n)}/{len(all_n)}")
    print(f"  Avg reprojection error (px): {np.mean(valid_err):.2f}" if valid_err else "  No matches")
    print(f"  Avg % correct < 5px:         {np.mean(all_5):.1f}%")
    print(f"  Avg % correct < 10px:        {np.mean(all_10):.1f}%")
    print(f"{'='*55}")

evaluate_model(model_pretrained, "Pretrained JamMa (baseline)")
evaluate_model(model_finetuned,  "Fine-tuned JamMa v2 (lower LR, 100 epochs)")

dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
test_images  = sorted(os.listdir(f'{dataset_root}/test/visible'))
os.makedirs('/kaggle/working/')

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/JamMa')

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from src.config.default import get_cfg_defaults
from src.lightning.lightning_jamma import PL_JamMa
from src.datasets.megadepth import read_megadepth_color

config = get_cfg_defaults()
config.merge_from_file('/kaggle/working/JamMa/configs/jamma/outdoor/final.py')

model = PL_JamMa(config,
    pretrained_ckpt='/kaggle/working/jamma_log/roadscene_finetune/version_0/checkpoints/last.ckpt')
model.eval().cuda()

dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
test_images  = sorted(os.listdir(f'{dataset_root}/test/visible'))
os.makedirs('/kaggle/working/match_viz', exist_ok=True)

results = []

for stem in test_images:
    vis_path = f'{dataset_root}/test/visible/{stem}'
    ir_path  = f'{dataset_root}/test/infrared/{stem}'

    img0, scale0, mask0, _ = read_megadepth_color(vis_path, 640, 8, padding=True)
    img1, scale1, mask1, _ = read_megadepth_color(ir_path,  640, 8, padding=True)
    H0, W0 = img0.shape[2], img0.shape[3]

    batch = {
        'imagec_0': img0.cuda(),
        'imagec_1': img1.cuda(),
        'h_8': H0 // 8,
        'w_8': W0 // 8,
        'dataset_name': ['RoadScene'],
        'pair_names': [(stem, stem)],
    }

    with torch.no_grad():
        with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
            model.backbone(batch)
        model.matcher(batch, mode='test')

    mkpts0 = batch['mkpts0_f'].cpu().numpy()
    mkpts1 = batch['mkpts1_f'].cpu().numpy()
    mconf  = batch['mconf'].cpu().numpy()

    # ── Reprojection error (since images are aligned, matches should be ~identity) ──
    if len(mkpts0) > 0:
        # Scale back to original image coordinates
        pts0 = mkpts0 / np.array([scale0[0], scale0[1]])
        pts1 = mkpts1 / np.array([scale1[0], scale1[1]])
        reproj_errors = np.linalg.norm(pts0 - pts1, axis=1)
        mean_err = reproj_errors.mean()
        pct_correct = (reproj_errors < 5).mean() * 100  # % matches within 5px
    else:
        mean_err, pct_correct = float('nan'), 0.0

    results.append({
        'stem': stem,
        'n_matches': len(mkpts0),
        'mean_reproj_err_px': mean_err,
        'pct_correct_5px': pct_correct,
        'mean_conf': mconf.mean() if len(mconf) > 0 else 0.0,
    })

    if test_images.index(stem) < 10:
        img0_vis = cv2.cvtColor(cv2.imread(vis_path), cv2.COLOR_BGR2RGB)
        img1_vis = cv2.cvtColor(cv2.imread(ir_path),  cv2.COLOR_BGR2RGB)
        h = max(img0_vis.shape[0], img1_vis.shape[0])
        canvas = np.zeros((h, img0_vis.shape[1] + img1_vis.shape[1], 3), dtype=np.uint8)
        canvas[:img0_vis.shape[0], :img0_vis.shape[1]] = img0_vis
        canvas[:img1_vis.shape[0], img0_vis.shape[1]:] = img1_vis

        for pt0, pt1, conf in zip(mkpts0[:200], mkpts1[:200], mconf[:200]):
            pt0i = (int(pt0[0] / scale0[0]), int(pt0[1] / scale0[1]))
            pt1i = (int(pt1[0] / scale1[0]) + img0_vis.shape[1], int(pt1[1] / scale1[1]))
            # Color by confidence: green=high, red=low
            c = int(conf * 255)
            color = (255 - c, c, 0)
            cv2.line(canvas, pt0i, pt1i, color, 1)
            cv2.circle(canvas, pt0i, 3, color, -1)
            cv2.circle(canvas, pt1i, 3, color, -1)

        out_path = f'/kaggle/working/match_viz/{stem}'
        cv2.imwrite(out_path, cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))

n_matches   = [r['n_matches'] for r in results]
reproj_errs = [r['mean_reproj_err_px'] for r in results if not np.isnan(r['mean_reproj_err_px'])]
pct_correct = [r['pct_correct_5px'] for r in results]

print(f"\n{'='*50}")
print(f"Results over {len(results)} test pairs")
print(f"{'='*50}")
print(f"Avg matches per pair:        {np.mean(n_matches):.1f}")
print(f"Pairs with >0 matches:       {sum(n>0 for n in n_matches)}/{len(n_matches)}")
print(f"Avg reprojection error (px): {np.mean(reproj_errs):.2f}")
print(f"Avg % correct matches <5px:  {np.mean(pct_correct):.1f}%")
print(f"\nVisualizations saved to /kaggle/working/match_viz/")

In [ ]:
model_pretrained = PL_JamMa(config,
    pretrained_ckpt='/kaggle/working/JamMa/weights/jamma.ckpt')

model_finetuned = PL_JamMa(config,
    pretrained_ckpt='/kaggle/working/jamma_log/roadscene_finetune/version_0/checkpoints/last.ckpt')

In [ ]:
def load_image_for_jamma(path, target_H, target_W):
    """Load and resize image to exact target dimensions."""
    img = cv2.imread(path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    img_resized = cv2.resize(img_rgb, (target_W, target_H))
    tensor = torch.from_numpy(img_resized).permute(2, 0, 1).float()[None] / 255.
    scale_x = target_W / W
    scale_y = target_H / H
    return tensor, (scale_x, scale_y), img_rgb

def get_target_size(path, target_size=640, df=8):
    """Compute target H, W from image dimensions."""
    img = cv2.imread(path)
    H, W = img.shape[:2]
    scale = target_size / max(H, W)
    new_H = int(round(H * scale / df)) * df
    new_W = int(round(W * scale / df)) * df
    return new_H, new_W

dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
test_images  = sorted(os.listdir(f'{dataset_root}/test/visible'))
os.makedirs('/kaggle/working/match_viz', exist_ok=True)

def evaluate_model(model, model_name):
    model.eval().cuda()
    results = []

    for stem in test_images:
        vis_path = f'{dataset_root}/test/visible/{stem}'
        ir_path  = f'{dataset_root}/test/infrared/{stem}'

        # Use visible image dimensions as the reference size for both
        target_H, target_W = get_target_size(vis_path)

        img0, scale0, vis_orig = load_image_for_jamma(vis_path, target_H, target_W)
        img1, scale1, ir_orig  = load_image_for_jamma(ir_path,  target_H, target_W)
        H0, W0 = img0.shape[2], img0.shape[3]

        batch = {
            'imagec_0': img0.cuda(),
            'imagec_1': img1.cuda(),
            'h_8': H0 // 8,
            'w_8': W0 // 8,
            'dataset_name': ['RoadScene'],
            'pair_names': [(stem, stem)],
        }

        with torch.no_grad():
            with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
                model.backbone(batch)
            model.matcher(batch, mode='test')

        mkpts0 = batch['mkpts0_f'].cpu().numpy()
        mkpts1 = batch['mkpts1_f'].cpu().numpy()

        if len(mkpts0) > 0:
            pts0 = mkpts0 / np.array([scale0[0], scale0[1]])
            pts1 = mkpts1 / np.array([scale1[0], scale1[1]])
            reproj = np.linalg.norm(pts0 - pts1, axis=1)
            mean_err = reproj.mean()
            pct_5px  = (reproj < 5).mean() * 100
            pct_10px = (reproj < 10).mean() * 100
        else:
            mean_err = float('nan')
            pct_5px = pct_10px = 0.0

        results.append({
            'stem': stem,
            'n_matches': len(mkpts0),
            'mean_reproj_err_px': mean_err,
            'pct_correct_5px': pct_5px,
            'pct_correct_10px': pct_10px,
            'mkpts0': mkpts0, 'mkpts1': mkpts1,
            'scale0': scale0, 'scale1': scale1,
            'vis_orig': vis_orig, 'ir_orig': ir_orig,
        })

    n_matches   = [r['n_matches'] for r in results]
    reproj_errs = [r['mean_reproj_err_px'] for r in results
                   if not np.isnan(r['mean_reproj_err_px'])]
    pct_5       = [r['pct_correct_5px'] for r in results]
    pct_10      = [r['pct_correct_10px'] for r in results]

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Avg matches per pair:         {np.mean(n_matches):.1f}")
    print(f"  Pairs with >0 matches:        {sum(n>0 for n in n_matches)}/{len(n_matches)}")
    print(f"  Avg reprojection error (px):  {np.mean(reproj_errs):.2f}" if reproj_errs else "  No matches")
    print(f"  Avg % correct < 5px:          {np.mean(pct_5):.1f}%")
    print(f"  Avg % correct < 10px:         {np.mean(pct_10):.1f}%")
    print(f"{'='*55}")
    return results

results_pretrained = evaluate_model(model_pretrained, "Pretrained JamMa (baseline)")
results_finetuned  = evaluate_model(model_finetuned,  "Fine-tuned JamMa (RoadScene)")

# Visualize first 5 pairs
for r in results_finetuned[:5]:
    stem = r['stem']
    mkpts0, mkpts1 = r['mkpts0'], r['mkpts1']
    scale0, scale1 = r['scale0'], r['scale1']
    img0_vis, img1_vis = r['vis_orig'], r['ir_orig']

    h = max(img0_vis.shape[0], img1_vis.shape[0])
    canvas = np.zeros((h, img0_vis.shape[1] + img1_vis.shape[1], 3), dtype=np.uint8)
    canvas[:img0_vis.shape[0], :img0_vis.shape[1]] = img0_vis
    canvas[:img1_vis.shape[0], img0_vis.shape[1]:] = img1_vis

    if len(mkpts0) > 0:
        pts0 = mkpts0 / np.array([scale0[0], scale0[1]])
        pts1 = mkpts1 / np.array([scale1[0], scale1[1]])
        reproj = np.linalg.norm(pts0 - pts1, axis=1)
        for pt0, pt1, err in zip(pts0[:300], pts1[:300], reproj[:300]):
            pt0i = (int(pt0[0]), int(pt0[1]))
            pt1i = (int(pt1[0]) + img0_vis.shape[1], int(pt1[1]))
            color = (0, 255, 0) if err < 10 else (255, 0, 0)
            cv2.line(canvas, pt0i, pt1i, color, 1)
            cv2.circle(canvas, pt0i, 3, color, -1)
            cv2.circle(canvas, pt1i, 3, color, -1)

    plt.figure(figsize=(18, 5))
    plt.imshow(canvas)
    plt.title(f'Fine-tuned — {len(mkpts0)} matches | {stem} (green=<10px, red=>10px)')
    plt.axis('off')
    plt.savefig(f'/kaggle/working/match_viz/fixed_{stem}', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
import os
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.datasets.megadepth import read_megadepth_color

dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
test_images  = sorted(os.listdir(f'{dataset_root}/test/visible'))
os.makedirs('/kaggle/working/match_viz', exist_ok=True)

def evaluate_model(model, model_name):
    model.eval().cuda()
    results = []

    for stem in test_images:
        vis_path = f'{dataset_root}/test/visible/{stem}'
        ir_path  = f'{dataset_root}/test/infrared/{stem}'

        img0, scale0, _, _ = read_megadepth_color(vis_path, 640, 8, padding=True)
        img1, scale1, _, _ = read_megadepth_color(ir_path,  640, 8, padding=True)
        H0, W0 = img0.shape[2], img0.shape[3]

        batch = {
            'imagec_0': img0.cuda(),
            'imagec_1': img1.cuda(),
            'h_8': H0 // 8,
            'w_8': W0 // 8,
            'dataset_name': ['RoadScene'],
            'pair_names': [(stem, stem)],
        }

        with torch.no_grad():
            with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
                model.backbone(batch)
            model.matcher(batch, mode='test')

        mkpts0 = batch['mkpts0_f'].cpu().numpy()
        mkpts1 = batch['mkpts1_f'].cpu().numpy()

        if len(mkpts0) > 0:
            pts0 = mkpts0 / np.array([scale0[0], scale0[1]])
            pts1 = mkpts1 / np.array([scale1[0], scale1[1]])
            reproj_errors = np.linalg.norm(pts0 - pts1, axis=1)
            mean_err   = reproj_errors.mean()
            pct_5px    = (reproj_errors < 5).mean() * 100
            pct_10px   = (reproj_errors < 10).mean() * 100
        else:
            mean_err = float('nan')
            pct_5px  = 0.0
            pct_10px = 0.0

        results.append({
            'stem': stem,
            'n_matches': len(mkpts0),
            'mean_reproj_err_px': mean_err,
            'pct_correct_5px': pct_5px,
            'pct_correct_10px': pct_10px,
        })

    n_matches   = [r['n_matches'] for r in results]
    reproj_errs = [r['mean_reproj_err_px'] for r in results if not np.isnan(r['mean_reproj_err_px'])]
    pct_5       = [r['pct_correct_5px'] for r in results]
    pct_10      = [r['pct_correct_10px'] for r in results]

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Avg matches per pair:         {np.mean(n_matches):.1f}")
    print(f"  Pairs with >0 matches:        {sum(n>0 for n in n_matches)}/{len(n_matches)}")
    print(f"  Avg reprojection error (px):  {np.mean(reproj_errs):.2f}")
    print(f"  Avg % correct < 5px:          {np.mean(pct_5):.1f}%")
    print(f"  Avg % correct < 10px:         {np.mean(pct_10):.1f}%")
    print(f"{'='*55}")

    return results

results_pretrained = evaluate_model(model_pretrained, "Pretrained JamMa (baseline)")
results_finetuned  = evaluate_model(model_finetuned,  "Fine-tuned JamMa (RoadScene)")

In [ ]:
stem = 'FLIR_00288.jpg'
vis_path = f'{dataset_root}/test/visible/{stem}'
ir_path  = f'{dataset_root}/test/infrared/{stem}'

img0, scale0, _, _ = read_megadepth_color(vis_path, 640, 8, padding=True)
img1, scale1, _, _ = read_megadepth_color(ir_path,  640, 8, padding=True)
H0, W0 = img0.shape[2], img0.shape[3]

batch = {
    'imagec_0': img0.cuda(), 'imagec_1': img1.cuda(),
    'h_8': H0 // 8, 'w_8': W0 // 8,
    'dataset_name': ['RoadScene'], 'pair_names': [(stem, stem)],
}
with torch.no_grad():
    with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
        model_finetuned.backbone(batch)
    model_finetuned.matcher(batch, mode='test')

mkpts0 = batch['mkpts0_f'].cpu().numpy()
mkpts1 = batch['mkpts1_f'].cpu().numpy()

pts0 = mkpts0 / np.array([scale0[0], scale0[1]])
pts1 = mkpts1 / np.array([scale1[0], scale1[1]])
reproj = np.linalg.norm(pts0 - pts1, axis=1)

print(f"Matches: {len(mkpts0)}")
print(f"Mean reprojection error: {reproj.mean():.2f} px")
print(f"Median reprojection error: {np.median(reproj):.2f} px")
print(f"% correct < 5px:  {(reproj < 5).mean()*100:.1f}%")
print(f"% correct < 10px: {(reproj < 10).mean()*100:.1f}%")
print(f"% correct < 20px: {(reproj < 20).mean()*100:.1f}%")

# Better visualization — draw on correctly scaled images
img0_vis = cv2.cvtColor(cv2.imread(vis_path), cv2.COLOR_BGR2RGB)
img1_vis = cv2.cvtColor(cv2.imread(ir_path),  cv2.COLOR_BGR2RGB)

h = max(img0_vis.shape[0], img1_vis.shape[0])
canvas = np.zeros((h, img0_vis.shape[1] + img1_vis.shape[1], 3), dtype=np.uint8)
canvas[:img0_vis.shape[0], :img0_vis.shape[1]] = img0_vis
canvas[:img1_vis.shape[0], img0_vis.shape[1]:] = img1_vis

# Color by error: green=correct, red=wrong
for pt0, pt1, err in zip(pts0[:200], pts1[:200], reproj[:200]):
    pt0i = (int(pt0[0]), int(pt0[1]))
    pt1i = (int(pt1[0]) + img0_vis.shape[1], int(pt1[1]))
    color = (0, 255, 0) if err < 10 else (255, 0, 0)
    cv2.line(canvas, pt0i, pt1i, color, 1)
    cv2.circle(canvas, pt0i, 3, color, -1)
    cv2.circle(canvas, pt1i, 3, color, -1)

plt.figure(figsize=(18, 6))
plt.imshow(canvas)
plt.title(f'{stem} — {len(mkpts0)} matches (green=<10px error, red=>10px)')
plt.axis('off')
plt.savefig('/kaggle/working/match_viz/quality_check.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def get_target_size(path, target_size=640, df=16):
    """Compute target H, W divisible by df=16 for JEGO scan compatibility."""
    img = cv2.imread(path)
    H, W = img.shape[:2]
    scale = target_size / max(H, W)
    new_H = int(round(H * scale / df)) * df
    new_W = int(round(W * scale / df)) * df
    return new_H, new_W

In [ ]:
# Inspect where matches actually are
stem = 'FLIR_01932.jpg'
vis_path = f'{dataset_root}/test/visible/{stem}'
ir_path  = f'{dataset_root}/test/infrared/{stem}'

target_H, target_W = get_target_size(vis_path)
img0, scale0, vis_orig = load_image_for_jamma(vis_path, target_H, target_W)
img1, scale1, ir_orig  = load_image_for_jamma(ir_path,  target_H, target_W)
H0, W0 = img0.shape[2], img0.shape[3]

batch = {
    'imagec_0': img0.cuda(), 'imagec_1': img1.cuda(),
    'h_8': H0 // 8, 'w_8': W0 // 8,
    'dataset_name': ['RoadScene'], 'pair_names': [(stem, stem)],
}
with torch.no_grad():
    with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
        model_finetuned.backbone(batch)
    model_finetuned.matcher(batch, mode='test')

mkpts0 = batch['mkpts0_f'].cpu().numpy()
mkpts1 = batch['mkpts1_f'].cpu().numpy()

print(f"Total matches: {len(mkpts0)}")
print(f"mkpts0 x range: {mkpts0[:,0].min():.1f} – {mkpts0[:,0].max():.1f}")
print(f"mkpts0 y range: {mkpts0[:,1].min():.1f} – {mkpts0[:,1].max():.1f}")
print(f"mkpts1 x range: {mkpts1[:,0].min():.1f} – {mkpts1[:,0].max():.1f}")
print(f"mkpts1 y range: {mkpts1[:,1].min():.1f} – {mkpts1[:,1].max():.1f}")
print(f"\nImage tensor size: {H0}x{W0}")
print(f"First 5 match points:")
for i in range(5):
    print(f"  ({mkpts0[i,0]:.1f}, {mkpts0[i,1]:.1f}) -> ({mkpts1[i,0]:.1f}, {mkpts1[i,1]:.1f})")

In [ ]:
stem = 'FLIR_01932.jpg'
vis_path = f'{dataset_root}/test/visible/{stem}'
ir_path  = f'{dataset_root}/test/infrared/{stem}'

target_H, target_W = get_target_size(vis_path)
img0, scale0, vis_orig = load_image_for_jamma(vis_path, target_H, target_W)
img1, scale1, ir_orig  = load_image_for_jamma(ir_path,  target_H, target_W)
H0, W0 = img0.shape[2], img0.shape[3]

batch = {
    'imagec_0': img0.cuda(), 'imagec_1': img1.cuda(),
    'h_8': H0 // 8, 'w_8': W0 // 8,
    'dataset_name': ['RoadScene'], 'pair_names': [(stem, stem)],
}
with torch.no_grad():
    with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
        model_finetuned.backbone(batch)
    model_finetuned.matcher(batch, mode='test')

mkpts0 = batch['mkpts0_f'].cpu().numpy()
mkpts1 = batch['mkpts1_f'].cpu().numpy()

# Draw on RESIZED images (same coordinate space as mkpts)
vis_resized = cv2.cvtColor(cv2.resize(
    cv2.imread(vis_path), (target_W, target_H)), cv2.COLOR_BGR2RGB)
ir_resized  = cv2.cvtColor(cv2.resize(
    cv2.imread(ir_path),  (target_W, target_H)), cv2.COLOR_BGR2RGB)

canvas = np.zeros((target_H, target_W * 2, 3), dtype=np.uint8)
canvas[:, :target_W] = vis_resized
canvas[:, target_W:] = ir_resized

reproj = np.linalg.norm(mkpts0 - mkpts1, axis=1)
print(f"Matches: {len(mkpts0)}")
print(f"Mean reproj error (tensor coords): {reproj.mean():.2f} px")
print(f"% correct < 5px:  {(reproj < 5).mean()*100:.1f}%")
print(f"% correct < 10px: {(reproj < 10).mean()*100:.1f}%")

# Sample 300 matches spread across image
idx = np.random.choice(len(mkpts0), min(300, len(mkpts0)), replace=False)
for i in idx:
    pt0 = (int(mkpts0[i, 0]), int(mkpts0[i, 1]))
    pt1 = (int(mkpts1[i, 0]) + target_W, int(mkpts1[i, 1]))
    color = (0, 255, 0) if reproj[i] < 20 else (255, 0, 0)
    cv2.line(canvas, pt0, pt1, color, 1)
    cv2.circle(canvas, pt0, 3, color, -1)
    cv2.circle(canvas, pt1, 3, color, -1)

plt.figure(figsize=(18, 6))
plt.imshow(canvas)
plt.title(f'{stem} — {len(mkpts0)} matches (green=<10px, red=>10px)')
plt.axis('off')
plt.savefig('/kaggle/working/match_viz/correct_viz.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
stem = 'FLIR_01932.jpg'
vis_path = f'{dataset_root}/test/visible/{stem}'
ir_path  = f'{dataset_root}/test/infrared/{stem}'

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

for ax, model, name in zip(axes,
                            [model_pretrained, model_finetuned],
                            ['Pretrained (baseline)', 'Fine-tuned']):
    model.eval().cuda()
    mkpts0, mkpts1, tH, tW, _, _ = run_inference(model, vis_path, ir_path)

    vis_r = cv2.cvtColor(cv2.resize(cv2.imread(vis_path), (tW, tH)), cv2.COLOR_BGR2RGB)
    ir_r  = cv2.cvtColor(cv2.resize(cv2.imread(ir_path),  (tW, tH)), cv2.COLOR_BGR2RGB)
    canvas = np.zeros((tH, tW * 2, 3), dtype=np.uint8)
    canvas[:, :tW] = vis_r
    canvas[:, tW:] = ir_r

    reproj = np.linalg.norm(mkpts0 - mkpts1, axis=1) if len(mkpts0) > 0 else np.array([])
    idx = np.random.choice(len(mkpts0), min(200, len(mkpts0)), replace=False) if len(mkpts0) > 0 else []
    for i in idx:
        pt0 = (int(mkpts0[i, 0]), int(mkpts0[i, 1]))
        pt1 = (int(mkpts1[i, 0]) + tW, int(mkpts1[i, 1]))
        color = (0, 255, 0) if reproj[i] < 10 else (255, 0, 0)
        cv2.line(canvas, pt0, pt1, color, 1)
        cv2.circle(canvas, pt0, 3, color, -1)
        cv2.circle(canvas, pt1, 3, color, -1)

    pct = (reproj < 10).mean() * 100 if len(reproj) > 0 else 0
    ax.imshow(canvas)
    ax.set_title(f'{name} — {len(mkpts0)} matches | {pct:.1f}% correct <10px')
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/match_viz/pretrained_vs_finetuned.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
def evaluate_and_visualize(model, model_name, test_images, dataset_root,
                            n_viz=5, error_thr=10, save_dir='/kaggle/working/match_viz'):
    model.eval().cuda()
    os.makedirs(save_dir, exist_ok=True)
    all_n, all_n_inliers, all_5, all_10, all_20 = [], [], [], [], []

    for i, stem in enumerate(test_images):
        vis_path = f'{dataset_root}/test/visible/{stem}'
        ir_path  = f'{dataset_root}/test/infrared/{stem}'

        try:
            target_H, target_W = get_target_size(vis_path)
            img0, _, _ = load_image_for_jamma(vis_path, target_H, target_W)
            img1, _, _ = load_image_for_jamma(ir_path,  target_H, target_W)
            H0, W0 = img0.shape[2], img0.shape[3]

            batch = {
                'imagec_0': img0.cuda(), 'imagec_1': img1.cuda(),
                'h_8': H0 // 8, 'w_8': W0 // 8,
                'dataset_name': ['RoadScene'], 'pair_names': [(stem, stem)],
            }
            with torch.no_grad():
                with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
                    model.backbone(batch)
                model.matcher(batch, mode='test')

            mkpts0 = batch['mkpts0_f'].cpu().numpy()
            mkpts1 = batch['mkpts1_f'].cpu().numpy()

            # ── RANSAC filtering ─────────────────────────────────────────
            if len(mkpts0) >= 8:
                _, mask = cv2.findHomography(
                    mkpts0, mkpts1,
                    cv2.RANSAC, ransacReprojThreshold=8.0,
                    confidence=0.999, maxIters=10000)
                mask = mask.ravel().astype(bool) if mask is not None else np.zeros(len(mkpts0), bool)
                mkpts0_in = mkpts0[mask]
                mkpts1_in = mkpts1[mask]
            else:
                mkpts0_in = mkpts0
                mkpts1_in = mkpts1
                mask = np.ones(len(mkpts0), bool)

            if len(mkpts0_in) > 0:
                reproj = np.linalg.norm(mkpts0_in - mkpts1_in, axis=1)
                all_5.append((reproj < 5).mean() * 100)
                all_10.append((reproj < 10).mean() * 100)
                all_20.append((reproj < 20).mean() * 100)
            else:
                reproj = np.array([])
                all_5.append(0); all_10.append(0); all_20.append(0)

            all_n.append(len(mkpts0))
            all_n_inliers.append(len(mkpts0_in))

            # ── Visualize first n_viz pairs ───────────────────────────────
            if i < n_viz:
                vis_r = cv2.cvtColor(cv2.resize(cv2.imread(vis_path),
                                     (target_W, target_H)), cv2.COLOR_BGR2RGB)
                ir_r  = cv2.cvtColor(cv2.resize(cv2.imread(ir_path),
                                     (target_W, target_H)), cv2.COLOR_BGR2RGB)
                canvas = np.zeros((target_H, target_W * 2, 3), dtype=np.uint8)
                canvas[:, :target_W] = vis_r
                canvas[:, target_W:] = ir_r

                # Draw only RANSAC inliers in green
                idx = np.random.choice(len(mkpts0_in),
                                       min(300, len(mkpts0_in)),
                                       replace=False) if len(mkpts0_in) > 0 else []
                for j in idx:
                    pt0 = (int(mkpts0_in[j, 0]), int(mkpts0_in[j, 1]))
                    pt1 = (int(mkpts1_in[j, 0]) + target_W, int(mkpts1_in[j, 1]))
                    cv2.line(canvas, pt0, pt1, (0, 255, 0), 1)
                    cv2.circle(canvas, pt0, 3, (0, 255, 0), -1)
                    cv2.circle(canvas, pt1, 3, (0, 255, 0), -1)

                pct5  = (reproj < 5).mean()  * 100 if len(reproj) > 0 else 0
                pct10 = (reproj < 10).mean() * 100 if len(reproj) > 0 else 0
                pct20 = (reproj < 20).mean() * 100 if len(reproj) > 0 else 0
                plt.figure(figsize=(18, 5))
                plt.imshow(canvas)
                plt.title(f'{model_name} | {stem}\n'
                          f'{len(mkpts0)} raw → {len(mkpts0_in)} inliers after RANSAC | '
                          f'<5px: {pct5:.1f}%  <10px: {pct10:.1f}%  <20px: {pct20:.1f}%')
                plt.axis('off')
                plt.savefig(f'{save_dir}/{model_name.replace(" ", "_")}_{stem}',
                            dpi=120, bbox_inches='tight')
                plt.show()

        except Exception as e:
            print(f"Error on {stem}: {e}")
            all_n.append(0); all_n_inliers.append(0)
            all_5.append(0); all_10.append(0); all_20.append(0)

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  Avg raw matches per pair:      {np.mean(all_n):.1f}")
    print(f"  Avg inliers after RANSAC:      {np.mean(all_n_inliers):.1f}")
    print(f"  Pairs with >0 inliers:         {sum(n>0 for n in all_n_inliers)}/{len(all_n_inliers)}")
    print(f"  Avg % correct < 5px:           {np.mean(all_5):.1f}%")
    print(f"  Avg % correct < 10px:          {np.mean(all_10):.1f}%")
    print(f"  Avg % correct < 20px:          {np.mean(all_20):.1f}%")
    print(f"{'='*60}\n")

# Run
dataset_root = '/kaggle/working/JamMa/src/datasets/RoadScene dataset'
test_images  = sorted(os.listdir(f'{dataset_root}/test/visible'))

evaluate_and_visualize(model_pretrained, "Pretrained",   test_images, dataset_root)
evaluate_and_visualize(model_finetuned,  "Finetuned_v2", test_images, dataset_root)

In [ ]:
stem = 'FLIR_00311.jpg'
vis_path = f'{dataset_root}/test/visible/{stem}'
ir_path  = f'{dataset_root}/test/infrared/{stem}'

target_H, target_W = get_target_size(vis_path)
img0, _, _ = load_image_for_jamma(vis_path, target_H, target_W)
H0, W0 = img0.shape[2], img0.shape[3]

print(f"Target size: {target_H} x {target_W}")
print(f"Tensor size: {H0} x {W0}")
print(f"h_8={H0//8}, w_8={W0//8}")
print(f"h_8 * w_8 = {(H0//8) * (W0//8)} coarse features")

# Check where matches actually are
import glob
# reload with the correct pipeline
img1, _, _ = load_image_for_jamma(ir_path, target_H, target_W)
batch = {
    'imagec_0': img0.cuda(), 'imagec_1': img1.cuda(),
    'h_8': H0 // 8, 'w_8': W0 // 8,
    'dataset_name': ['RoadScene'], 'pair_names': [(stem, stem)],
}
with torch.no_grad():
    with torch.autocast(enabled=config.JAMMA.MP, device_type='cuda'):
        model_finetuned.backbone(batch)
    model_finetuned.matcher(batch, mode='test')

mkpts0 = batch['mkpts0_f'].cpu().numpy()
print(f"\nMatches: {len(mkpts0)}")
print(f"X range: {mkpts0[:,0].min():.0f} – {mkpts0[:,0].max():.0f} (image width: {target_W})")
print(f"Y range: {mkpts0[:,1].min():.0f} – {mkpts0[:,1].max():.0f} (image height: {target_H})")
print(f"Matches in left half (x < {target_W//2}): {(mkpts0[:,0] < target_W//2).sum()}")
print(f"Matches in right half (x >= {target_W//2}): {(mkpts0[:,0] >= target_W//2).sum()}")